# 03 Retinotopic organisation in the reference tractogram (H1)
Checks that the measure detects the known organisation on full-quality data: per-streamline retinotopic endpoint error against measured pRF maps, compared with distance-matched surrogates (RSI), the split-half noise ceiling and a proximity-only lower bound.

In [ ]:
DEV_MODE = True    # True: small test run (dev: block of config.yaml). False: full experiment (full: block).

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from retinopath.core import load_config, get_device, seed_all, Timer
from retinopath import pipeline as PL

cfg = load_config(DEV_MODE); device = get_device(); seed_all(cfg.seed)
RES = cfg.P.out / "results"; FIG = cfg.P.out / "figures"; RES.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print(f"DEV_MODE={DEV_MODE} | device={device} | outputs -> {cfg.P.out}")
if DEV_MODE: print("DEV_MODE: small test run. Its outputs are kept apart in outputs/dev and are not experiment results.")

In [ ]:
subjects = PL.make_subject_lists(cfg); test = [PL.session(s) for s in subjects["test"]]
with Timer(cfg, "03_reference_metric"):
    df = PL.run_grid(cfg, test, [("REF", "msmt_csd"), ("REF", "proximity")], "h1_reference")
cols = ["sid", "method", "n_pairs", "n_valid", "ree_median_all", "ree_null_median_all", "rsi_ree_all", "rsi_ree_geodesic_all", "noise_ceiling_deg"]
df[cols].round(3)

In [ ]:
# The same tractograms scored with two other keys: the HCP group-average pRF maps (template; the average includes
# the subject) and the anatomical atlas of Benson et al. 2014 (atlas; no functional data of the subject at all)
keys = {"rsi_ree_all": "measured pRF", "template_rsi_ree_all": "group-average pRF", "atlas_rsi_ree_all": "anatomical atlas"}
df.groupby("method")[list(keys)].median().rename(columns=keys).round(3)

In [ ]:
pairs = ["V1-V2", "V2-V3", "V1-V3"]; fig, ax = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for a, q, t in zip(ax, ["rsi_ree", "rsi_dlogecc", "rsi_dang"], ["visual-field distance", "eccentricity", "polar angle"]):
    for m, c in (("msmt_csd", "C0"), ("proximity", "0.6")):
        d = df[df.method == m][[f"{q}_{p}" for p in pairs]].to_numpy()
        a.plot(np.arange(3) + (0.1 if m == "proximity" else -0.1), d.T, "o", color=c, alpha=0.6)
    a.axhline(0, color="k", lw=0.8); a.set_xticks(range(3), pairs); a.set_title(f"RSI: {t}")
ax[0].set_ylabel("retinotopic specificity index"); fig.savefig(FIG / "h1_rsi.png", dpi=150, bbox_inches="tight")
print("blue = tractography, grey = proximity-only lower bound; 0 = no better than distance alone")